In [ ]:
"""
THI Correlation Analysis with Silkworm Diseases
================================================
Analyzes correlation between Thermo-Humidity Index and:
- PB (Pebrine)
- VR (Viriosis)
- BT (Bacteriosis)

Date Range: September 24 - October 30, 2024
Location: Ranchi, Jharkhand, India
"""

# ============================================================================
# SECTION 1: Installation and Imports
# ============================================================================

!pip install -q requests pandas matplotlib seaborn numpy scipy openpyxl

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from scipy.stats import pointbiserialr
import warnings
warnings.filterwarnings('ignore')

# Set high-quality plot parameters for 600 DPI
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 16
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['figure.titlesize'] = 22

print("✓ Libraries imported successfully!")
print("✓ High-resolution settings (600 DPI) configured")

# ============================================================================
# SECTION 2: Input Data
# ============================================================================

# Disease observation data
data = {
    'Date': ['24.09.24', '25.09.24', '26.09.24', '27.09.24', '28.09.24', '29.09.24',
             '30.09.24', '01.10.24', '02.10.24', '03.10.24', '06.10.24', '07.10.24',
             '14.10.24', '15.10.24', '16.10.24', '17.10.24', '18.10.24', '19.10.24',
             '20.10.24', '21.10.24', '22.10.24', '23.10.24', '24.10.24', '25.10.24',
             '26.10.24', '27.10.24', '28.10.24', '29.10.24', '30.10.24'],
    'PB': ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no',
           'no', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'yes', 'yes', 'no',
           'no', 'no', 'no', 'no', 'no', 'no'],
    'VR': ['yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'yes',
           'no', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'no', 'yes',
           'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no'],
    'BT': ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'no',
           'yes', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'no', 'no', 'no', 'no',
           'no', 'no', 'no', 'no', 'no', 'no']
}

# Disease full names
disease_names = {
    'PB': 'Pebrine',
    'VR': 'Viriosis',
    'BT': 'Bacteriosis'
}

# Create DataFrame
df_disease = pd.DataFrame(data)

# Convert date to proper format
df_disease['Date'] = pd.to_datetime(df_disease['Date'], format='%d.%m.%y')

# Convert yes/no to binary (1/0)
for col in ['PB', 'VR', 'BT']:
    df_disease[f'{col}_binary'] = (df_disease[col] == 'yes').astype(int)

print("✓ Disease data loaded")
print(f"✓ Date range: {df_disease['Date'].min().date()} to {df_disease['Date'].max().date()}")
print(f"✓ Total observations: {len(df_disease)} days")
print("\nDisease Data Summary:")
for code, name in disease_names.items():
    count = df_disease[f'{code}_binary'].sum()
    print(f"  {code} ({name}): {count} positive cases")

# ============================================================================
# SECTION 3: Weather Data Functions
# ============================================================================

def fetch_weather_data_open_meteo(lat, lon, start_date, end_date):
    """
    Fetch historical weather data using Open-Meteo API
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "apparent_temperature"
        ],
        "timezone": "Asia/Kolkata"
    }

    print(f"\nFetching weather data...")
    print(f"  Location: ({lat}°N, {lon}°E)")
    print(f"  Date range: {start_date} to {end_date}")

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        hourly_data = data['hourly']

        df = pd.DataFrame({
            'datetime': pd.to_datetime(hourly_data['time']),
            'temperature_c': hourly_data['temperature_2m'],
            'relative_humidity': hourly_data['relative_humidity_2m'],
            'dew_point_c': hourly_data['dew_point_2m'],
            'apparent_temp_c': hourly_data['apparent_temperature']
        })

        print(f"✓ Successfully fetched {len(df)} hourly records")
        return df

    except Exception as e:
        print(f"✗ Error fetching data: {e}")
        return None


def calculate_thi_nrc(temp_c, rh):
    """
    Calculate Thermo-Humidity Index using NRC (1971) formula
    """
    temp_f = (1.8 * temp_c) + 32
    thi = temp_f - ((0.55 - 0.0055 * rh) * (temp_f - 58))
    return thi


def classify_thi_stress(thi):
    """
    Classify THI into stress categories
    """
    if thi < 70:
        return "No Stress"
    elif thi < 72:
        return "Mild Stress"
    elif thi < 80:
        return "Moderate Stress"
    elif thi < 90:
        return "Severe Stress"
    else:
        return "Emergency"


# ============================================================================
# SECTION 4: Fetch Weather Data for Ranchi
# ============================================================================

# Ranchi coordinates
RANCHI_LAT = 23.3441
RANCHI_LON = 85.3096

# Get date range from disease data
start_date = df_disease['Date'].min().strftime('%Y-%m-%d')
end_date = df_disease['Date'].max().strftime('%Y-%m-%d')

# Fetch weather data
df_weather = fetch_weather_data_open_meteo(RANCHI_LAT, RANCHI_LON, start_date, end_date)

if df_weather is not None:
    # Calculate THI
    df_weather['THI'] = df_weather.apply(
        lambda row: calculate_thi_nrc(row['temperature_c'], row['relative_humidity']),
        axis=1
    )
    df_weather['Stress_Level'] = df_weather['THI'].apply(classify_thi_stress)
    df_weather['Date'] = df_weather['datetime'].dt.date

    print("\n✓ THI calculated for all hourly records")

# ============================================================================
# SECTION 5: Daily Aggregation
# ============================================================================

if df_weather is not None:
    # Calculate daily statistics
    daily_weather = df_weather.groupby('Date').agg({
        'temperature_c': ['mean', 'min', 'max'],
        'relative_humidity': ['mean', 'min', 'max'],
        'THI': ['mean', 'min', 'max']
    }).round(2)

    # Flatten column names
    daily_weather.columns = ['_'.join(col).strip() for col in daily_weather.columns.values]
    daily_weather = daily_weather.reset_index()
    daily_weather['Date'] = pd.to_datetime(daily_weather['Date'])

    # Rename columns for clarity
    daily_weather.rename(columns={
        'temperature_c_mean': 'Temp_Mean',
        'temperature_c_min': 'Temp_Min',
        'temperature_c_max': 'Temp_Max',
        'relative_humidity_mean': 'RH_Mean',
        'relative_humidity_min': 'RH_Min',
        'relative_humidity_max': 'RH_Max',
        'THI_mean': 'THI_Mean',
        'THI_min': 'THI_Min',
        'THI_max': 'THI_Max'
    }, inplace=True)

    print("\n✓ Daily weather statistics calculated")

# ============================================================================
# SECTION 6: Merge Disease and Weather Data
# ============================================================================

if df_weather is not None:
    # Merge datasets
    df_merged = pd.merge(df_disease, daily_weather, on='Date', how='left')

    print("\n✓ Disease and weather data merged")
    print(f"✓ Final dataset: {len(df_merged)} observations")

    # Save merged data
    df_merged.to_csv('merged_disease_weather_data.csv', index=False)
    print("\n✓ Merged data saved to 'merged_disease_weather_data.csv'")

# ============================================================================
# SECTION 7: Statistical Analysis
# ============================================================================

def perform_correlation_analysis(df, variable_name):
    """
    Perform comprehensive correlation analysis for a disease variable
    """
    results = {
        'variable': variable_name,
        'disease_name': disease_names[variable_name],
        'observations': len(df),
        'positive_cases': df[f'{variable_name}_binary'].sum(),
        'negative_cases': len(df) - df[f'{variable_name}_binary'].sum()
    }

    # Point-biserial correlation (for binary vs continuous)
    valid_data = df[[f'{variable_name}_binary', 'THI_Mean']].dropna()

    if len(valid_data) > 0:
        corr_thi, p_value_thi = pointbiserialr(valid_data[f'{variable_name}_binary'],
                                                valid_data['THI_Mean'])
        results['correlation_thi'] = corr_thi
        results['p_value_thi'] = p_value_thi
        results['significant_thi'] = 'Yes' if p_value_thi < 0.05 else 'No'

        # Also correlate with temperature and humidity
        corr_temp, p_value_temp = pointbiserialr(valid_data[f'{variable_name}_binary'],
                                                  df.loc[valid_data.index, 'Temp_Mean'])
        results['correlation_temp'] = corr_temp
        results['p_value_temp'] = p_value_temp

        corr_rh, p_value_rh = pointbiserialr(valid_data[f'{variable_name}_binary'],
                                             df.loc[valid_data.index, 'RH_Mean'])
        results['correlation_rh'] = corr_rh
        results['p_value_rh'] = p_value_rh

        # Mean THI for positive vs negative cases
        results['thi_when_yes'] = df[df[f'{variable_name}_binary'] == 1]['THI_Mean'].mean()
        results['thi_when_no'] = df[df[f'{variable_name}_binary'] == 0]['THI_Mean'].mean()
        results['thi_difference'] = results['thi_when_yes'] - results['thi_when_no']

        # T-test
        yes_group = df[df[f'{variable_name}_binary'] == 1]['THI_Mean'].dropna()
        no_group = df[df[f'{variable_name}_binary'] == 0]['THI_Mean'].dropna()

        if len(yes_group) > 0 and len(no_group) > 0:
            t_stat, t_pvalue = stats.ttest_ind(yes_group, no_group)
            results['t_statistic'] = t_stat
            results['t_test_pvalue'] = t_pvalue

    return results


# Perform analysis for all variables (excluding FG)
print("\n" + "="*80)
print("STATISTICAL ANALYSIS")
print("="*80)

analysis_results = {}
variables = ['PB', 'VR', 'BT']

for var in variables:
    print(f"\n--- Analysis for {var} ({disease_names[var]}) ---")
    results = perform_correlation_analysis(df_merged, var)
    analysis_results[var] = results

    print(f"Observations: {results['observations']}")
    print(f"Positive cases (yes): {results['positive_cases']}")
    print(f"Negative cases (no): {results['negative_cases']}")
    print(f"\nCorrelation with THI: {results.get('correlation_thi', 'N/A'):.4f}")
    print(f"P-value: {results.get('p_value_thi', 'N/A'):.4f}")
    print(f"Significant (p<0.05): {results.get('significant_thi', 'N/A')}")
    print(f"\nMean THI when {var}=Yes: {results.get('thi_when_yes', 'N/A'):.2f}")
    print(f"Mean THI when {var}=No: {results.get('thi_when_no', 'N/A'):.2f}")
    print(f"Difference: {results.get('thi_difference', 'N/A'):.2f}")

# ============================================================================
# SECTION 8: Save Statistical Results to Text File
# ============================================================================

with open('statistical_analysis_results.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("THERMO-HUMIDITY INDEX AND SILKWORM DISEASE CORRELATION ANALYSIS\n")
    f.write("="*80 + "\n")
    f.write("Location: Ranchi, Jharkhand, India\n")
    f.write(f"Date Range: {start_date} to {end_date}\n")
    f.write(f"Total Observations: {len(df_merged)} days\n")
    f.write("="*80 + "\n\n")

    f.write("SILKWORM DISEASES:\n")
    for code, name in disease_names.items():
        f.write(f"  {code} - {name}\n")
    f.write("\n")

    f.write("THI STRESS CATEGORIES:\n")
    f.write("  < 70  : No Stress\n")
    f.write("  70-72 : Mild Stress\n")
    f.write("  72-80 : Moderate Stress\n")
    f.write("  80-90 : Severe Stress\n")
    f.write("  > 90  : Emergency\n\n")

    f.write("="*80 + "\n")
    f.write("STATISTICAL RESULTS\n")
    f.write("="*80 + "\n\n")

    for var in variables:
        results = analysis_results[var]
        f.write(f"\n{'='*80}\n")
        f.write(f"Disease: {results['disease_name']} ({var})\n")
        f.write(f"{'='*80}\n\n")

        f.write(f"Sample Size:\n")
        f.write(f"  Total observations: {results['observations']}\n")
        f.write(f"  Positive cases (disease present): {results['positive_cases']}\n")
        f.write(f"  Negative cases (disease absent): {results['negative_cases']}\n\n")

        f.write(f"Point-Biserial Correlation:\n")
        f.write(f"  Correlation with THI: {results.get('correlation_thi', 'N/A'):.4f}\n")
        f.write(f"  P-value: {results.get('p_value_thi', 'N/A'):.6f}\n")
        f.write(f"  Statistically Significant (p<0.05): {results.get('significant_thi', 'N/A')}\n\n")

        f.write(f"  Correlation with Temperature: {results.get('correlation_temp', 'N/A'):.4f}\n")
        f.write(f"  P-value: {results.get('p_value_temp', 'N/A'):.6f}\n\n")

        f.write(f"  Correlation with Humidity: {results.get('correlation_rh', 'N/A'):.4f}\n")
        f.write(f"  P-value: {results.get('p_value_rh', 'N/A'):.6f}\n\n")

        f.write(f"Mean THI Comparison:\n")
        f.write(f"  Mean THI when {results['disease_name']} present: {results.get('thi_when_yes', 'N/A'):.2f}\n")
        f.write(f"  Mean THI when {results['disease_name']} absent: {results.get('thi_when_no', 'N/A'):.2f}\n")
        f.write(f"  Difference: {results.get('thi_difference', 'N/A'):.2f}\n\n")

        if 't_test_pvalue' in results:
            f.write(f"Independent T-Test:\n")
            f.write(f"  T-statistic: {results['t_statistic']:.4f}\n")
            f.write(f"  P-value: {results['t_test_pvalue']:.6f}\n")
            f.write(f"  Significant difference (p<0.05): {'Yes' if results['t_test_pvalue'] < 0.05 else 'No'}\n\n")

        # Interpretation
        f.write(f"Interpretation:\n")
        corr = results.get('correlation_thi', 0)
        if abs(corr) < 0.1:
            strength = "negligible"
        elif abs(corr) < 0.3:
            strength = "weak"
        elif abs(corr) < 0.5:
            strength = "moderate"
        else:
            strength = "strong"

        direction = "positive" if corr > 0 else "negative"

        f.write(f"  There is a {strength} {direction} correlation between {results['disease_name']} and THI.\n")

        if results.get('significant_thi') == 'Yes':
            f.write(f"  This correlation is statistically significant (p<0.05).\n")
        else:
            f.write(f"  This correlation is NOT statistically significant (p≥0.05).\n")

        f.write("\n")

print("\n✓ Statistical results saved to 'statistical_analysis_results.txt'")

# ============================================================================
# SECTION 9: VISUALIZATION 1 - Combined Time Series
# ============================================================================

print("\nGenerating visualizations...")

fig = plt.figure(figsize=(22, 11))

# Create axes with space at top for legend
ax1 = plt.subplot(111)

# Define colors for each disease
disease_colors = {
    'PB': '#E74C3C',
    'VR': '#3498DB',
    'BT': '#F39C12'
}

# Plot THI as line
ax1.plot(df_merged['Date'], df_merged['THI_Mean'],
         color='black', linewidth=3.5, marker='o', markersize=8,
         label='THI', alpha=0.8, zorder=5)

# Add stress level zones with stronger colors (no Mild Stress zone)
ax1.axhspan(0, 70, alpha=0.25, color='green', label='No Stress (<70)')
ax1.axhspan(72, 80, alpha=0.25, color='orange', label='Moderate Stress (72-80)')
ax1.axhspan(80, 100, alpha=0.25, color='red', label='Severe Stress (>80)')

ax1.set_xlabel('Date', fontsize=22, fontweight='bold')
ax1.set_ylabel('Thermo-Humidity Index (THI)', fontsize=22, fontweight='bold', color='black')
ax1.tick_params(axis='y', labelcolor='black', labelsize=18)
ax1.tick_params(axis='x', rotation=45, labelsize=18)
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=1.5)

# Secondary y-axis for disease occurrence
ax2 = ax1.twinx()

# Plot each disease with more opaque lines
y_offset = 0
for var in variables:
    disease_data = df_merged[[f'{var}_binary', 'Date']].copy()
    disease_data[f'{var}_plot'] = disease_data[f'{var}_binary'] + y_offset

    # Plot line connecting disease occurrences - MORE OPAQUE
    ax2.plot(df_merged['Date'], disease_data[f'{var}_plot'],
             color=disease_colors[var], linewidth=3, alpha=0.7,
             linestyle='--', zorder=3)

    # Mark disease occurrences with large markers
    yes_dates = df_merged[df_merged[f'{var}_binary'] == 1]['Date']
    yes_values = [1 + y_offset] * len(yes_dates)

    ax2.scatter(yes_dates, yes_values,
                color=disease_colors[var], s=400, alpha=0.9,
                marker='s', edgecolor='black', linewidth=2.5,
                label=f'{disease_names[var]} ({var})', zorder=10)

    y_offset += 0.2

ax2.set_ylabel('Disease Occurrence', fontsize=22, fontweight='bold')
ax2.set_ylim(-0.3, 1.8)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Absent', 'Present'], fontsize=18)
ax2.tick_params(axis='y', labelsize=18)

# Get all legend elements
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

# Create single legend ABOVE the plot
fig.legend(lines1 + lines2, labels1 + labels2,
          loc='upper center', fontsize=15,
          framealpha=0.98, edgecolor='black',
          ncol=4, bbox_to_anchor=(0.5, 0.97),
          columnspacing=2)

# Adjust layout to make room for legend at top
plt.subplots_adjust(top=0.90, bottom=0.12, left=0.08, right=0.92)

plt.savefig('fig1_combined_timeseries.png', dpi=600, bbox_inches='tight', pad_inches=0.3)
plt.close()

print("✓ Saved: fig1_combined_timeseries.png")

# ============================================================================
# SECTION 10: VISUALIZATION 2 - 3x3 Correlation Matrix
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 9))

# Create 3x3 correlation matrix
climate_vars = ['THI_Mean', 'Temp_Mean', 'RH_Mean']
disease_vars = ['PB_binary', 'VR_binary', 'BT_binary']

# Calculate correlations
corr_matrix_3x3 = np.zeros((3, 3))

for i, climate_var in enumerate(climate_vars):
    for j, disease_var in enumerate(disease_vars):
        valid_data = df_merged[[climate_var, disease_var]].dropna()
        if len(valid_data) > 0:
            corr, _ = pointbiserialr(valid_data[disease_var], valid_data[climate_var])
            corr_matrix_3x3[i, j] = corr

# Create DataFrame with short names only
corr_df = pd.DataFrame(
    corr_matrix_3x3,
    index=['THI', 'Temperature (°C)', 'Humidity (%)'],
    columns=['PB', 'VR', 'BT']
)

# Create colormap
cmap = sns.diverging_palette(250, 10, as_cmap=True)

# Create heatmap
im = ax.imshow(corr_df.values, cmap=cmap, aspect='auto', vmin=-1, vmax=1)

# Set ticks and labels
ax.set_xticks(np.arange(3))
ax.set_yticks(np.arange(3))
ax.set_xticklabels(corr_df.columns, fontsize=24, fontweight='bold')
ax.set_yticklabels(corr_df.index, fontsize=24, fontweight='bold')

# Rotate x labels
plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

# Add correlation values and colored borders
for i in range(3):
    for j in range(3):
        corr_val = corr_df.iloc[i, j]

        # Text color
        if abs(corr_val) > 0.5:
            text_color = "white"
        else:
            text_color = "black"

        # Add value
        ax.text(j, i, f'{corr_val:.3f}',
                ha="center", va="center", color=text_color,
                fontsize=26, fontweight='bold')

        # Add colored border based on correlation strength
        if abs(corr_val) > 0.3:
            # Use the actual cell color for the border
            cell_color = cmap((corr_val + 1) / 2)
            rect = plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                  fill=False, edgecolor=cell_color,
                                  linewidth=6, linestyle='-')
            ax.add_patch(rect)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=20)
cbar.set_label('Correlation Coefficient', fontsize=22, fontweight='bold',
               rotation=270, labelpad=40)

# Title - simple and clean
ax.set_title('Climate Variables vs Silkworm Diseases\nColored borders: |r| > 0.3',
             fontsize=24, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('fig2_correlation_matrix.png', dpi=600, bbox_inches='tight')
plt.close()

print("✓ Saved: fig2_correlation_matrix.png")

# ============================================================================
# SECTION 11: Summary Statistics Table
# ============================================================================

# Create summary table
summary_data = []
for var in variables:
    res = analysis_results[var]
    summary_data.append({
        'Disease': disease_names[var],
        'Code': var,
        'Positive Cases': res['positive_cases'],
        'Negative Cases': res['negative_cases'],
        'Correlation (r)': f"{res.get('correlation_thi', 0):.4f}",
        'P-value': f"{res.get('p_value_thi', 1):.6f}",
        'Significant': res.get('significant_thi', 'N/A'),
        'THI (Disease Present)': f"{res.get('thi_when_yes', 0):.2f}",
        'THI (Disease Absent)': f"{res.get('thi_when_no', 0):.2f}",
        'Difference': f"{res.get('thi_difference', 0):.2f}"
    })

summary_df = pd.DataFrame(summary_data)

# Save to CSV
summary_df.to_csv('summary_statistics_table.csv', index=False)
print("\n✓ Saved: summary_statistics_table.csv")

# ============================================================================
# SECTION 12: Final Summary Report
# ============================================================================

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

print("\nGenerated Files:")
print("\nData Files:")
print("  1. merged_disease_weather_data.csv - Complete dataset")
print("  2. summary_statistics_table.csv - Statistical summary")
print("  3. statistical_analysis_results.txt - Detailed text report")

print("\nVisualization Files (600 DPI, Large Fonts):")
print("  1. fig1_combined_timeseries.png - Time series (legends outside)")
print("  2. fig2_correlation_matrix.png - 3x3 correlation heatmap")

print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

# Overall THI statistics
print(f"\nTHI Statistics for Study Period:")
print(f"  Mean THI: {df_merged['THI_Mean'].mean():.2f}")
print(f"  Min THI: {df_merged['THI_Mean'].min():.2f}")
print(f"  Max THI: {df_merged['THI_Mean'].max():.2f}")
print(f"  Standard Deviation: {df_merged['THI_Mean'].std():.2f}")

print("\nDisease-THI Correlations:")
print("-" * 80)
for var in variables:
    res = analysis_results[var]
    corr = res.get('correlation_thi', 0)
    p_val = res.get('p_value_thi', 1)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"

    print(f"\n{disease_names[var]} ({var}):")
    print(f"  Positive cases: {res['positive_cases']}/{res['observations']} days ({res['positive_cases']/res['observations']*100:.1f}%)")
    print(f"  Correlation: r = {corr:.4f} (p = {p_val:.6f}) {sig}")

    if abs(corr) < 0.1:
        strength = "Negligible"
    elif abs(corr) < 0.3:
        strength = "Weak"
    elif abs(corr) < 0.5:
        strength = "Moderate"
    else:
        strength = "Strong"

    direction = "positive" if corr > 0 else "negative"
    print(f"  {strength} {direction} correlation")

    print(f"  Mean THI when disease present: {res.get('thi_when_yes', 0):.2f}")
    print(f"  Mean THI when disease absent: {res.get('thi_when_no', 0):.2f}")
    print(f"  Difference: {res.get('thi_difference', 0):.2f}")

    if res.get('significant_thi') == 'Yes':
        print(f"  ✓ Statistically SIGNIFICANT relationship")
    else:
        print(f"  ✗ NOT statistically significant")

print("\n" + "="*80)
print("Legend:")
print("  *** p < 0.001 (Highly significant)")
print("  **  p < 0.01  (Very significant)")
print("  *   p < 0.05  (Significant)")
print("  ns  p ≥ 0.05  (Not significant)")
print("="*80)

# Climate summary
print("\nClimate Conditions During Study Period:")
print(f"  Temperature: {df_merged['Temp_Mean'].mean():.2f}°C (±{df_merged['Temp_Mean'].std():.2f})")
print(f"  Range: {df_merged['Temp_Mean'].min():.2f}°C to {df_merged['Temp_Mean'].max():.2f}°C")
print(f"  Humidity: {df_merged['RH_Mean'].mean():.2f}% (±{df_merged['RH_Mean'].std():.2f})")
print(f"  Range: {df_merged['RH_Mean'].min():.2f}% to {df_merged['RH_Mean'].max():.2f}%")

# Stress level distribution
print("\nTHI Stress Level Distribution:")
stress_counts = {}
for idx, row in df_merged.iterrows():
    level = classify_thi_stress(row['THI_Mean'])
    stress_counts[level] = stress_counts.get(level, 0) + 1

for level in ['No Stress', 'Mild Stress', 'Moderate Stress', 'Severe Stress', 'Emergency']:
    count = stress_counts.get(level, 0)
    pct = count / len(df_merged) * 100
    print(f"  {level}: {count} days ({pct:.1f}%)")

print("\n" + "="*80)
print("RECOMMENDATIONS:")
print("="*80)

# Generate recommendations
recommendations = []

for var in variables:
    res = analysis_results[var]
    if res.get('significant_thi') == 'Yes':
        corr = res.get('correlation_thi', 0)
        if corr > 0:
            recommendations.append(
                f"• {disease_names[var]} shows significant positive correlation with THI. "
                f"Increased monitoring recommended when THI > {res.get('thi_when_yes', 70):.1f}."
            )
        else:
            recommendations.append(
                f"• {disease_names[var]} shows significant negative correlation with THI. "
                f"Increased monitoring recommended when THI < {res.get('thi_when_yes', 70):.1f}."
            )

if len(recommendations) > 0:
    for rec in recommendations:
        print(f"\n{rec}")
else:
    print("\n• No statistically significant correlations found between THI and diseases.")
    print("• Continue regular monitoring regardless of THI levels.")

print("\n• Maintain optimal rearing conditions: Temperature 24-28°C, Humidity 70-85%")
print("• Implement preventive measures during periods of environmental stress")
print("• Regular disinfection and hygiene protocols are essential regardless of THI")

print("\n" + "="*80)

# Additional statistics table
print("\nDetailed Statistics Table:")
print("="*80)
print(summary_df.to_string(index=False))

# Save enhanced summary
with open('analysis_summary.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("SILKWORM DISEASE AND THI CORRELATION ANALYSIS - SUMMARY\n")
    f.write("="*80 + "\n\n")

    f.write(f"Study Location: Ranchi, Jharkhand, India\n")
    f.write(f"Coordinates: {RANCHI_LAT}°N, {RANCHI_LON}°E\n")
    f.write(f"Study Period: {start_date} to {end_date}\n")
    f.write(f"Total Observations: {len(df_merged)} days\n\n")

    f.write("="*80 + "\n")
    f.write("DISEASE DEFINITIONS\n")
    f.write("="*80 + "\n")
    for code, name in disease_names.items():
        f.write(f"{code} - {name}\n")
    f.write("\n")

    f.write("="*80 + "\n")
    f.write("KEY FINDINGS\n")
    f.write("="*80 + "\n\n")

    for var in variables:
        res = analysis_results[var]
        f.write(f"{disease_names[var]} ({var}):\n")
        f.write(f"  Incidence: {res['positive_cases']}/{res['observations']} days\n")
        f.write(f"  Correlation with THI: r = {res.get('correlation_thi', 0):.4f}\n")
        f.write(f"  P-value: {res.get('p_value_thi', 1):.6f}\n")
        f.write(f"  Statistical Significance: {res.get('significant_thi', 'N/A')}\n")
        f.write(f"  Mean THI (disease present): {res.get('thi_when_yes', 0):.2f}\n")
        f.write(f"  Mean THI (disease absent): {res.get('thi_when_no', 0):.2f}\n")
        f.write(f"  Difference: {res.get('thi_difference', 0):.2f}\n\n")

    f.write("="*80 + "\n")
    f.write("CLIMATE STATISTICS\n")
    f.write("="*80 + "\n\n")
    f.write(f"THI (Thermo-Humidity Index):\n")
    f.write(f"  Mean: {df_merged['THI_Mean'].mean():.2f}\n")
    f.write(f"  Standard Deviation: {df_merged['THI_Mean'].std():.2f}\n")
    f.write(f"  Range: {df_merged['THI_Mean'].min():.2f} - {df_merged['THI_Mean'].max():.2f}\n\n")

    f.write(f"Temperature:\n")
    f.write(f"  Mean: {df_merged['Temp_Mean'].mean():.2f}°C\n")
    f.write(f"  Standard Deviation: {df_merged['Temp_Mean'].std():.2f}°C\n")
    f.write(f"  Range: {df_merged['Temp_Mean'].min():.2f}°C - {df_merged['Temp_Mean'].max():.2f}°C\n\n")

    f.write(f"Relative Humidity:\n")
    f.write(f"  Mean: {df_merged['RH_Mean'].mean():.2f}%\n")
    f.write(f"  Standard Deviation: {df_merged['RH_Mean'].std():.2f}%\n")
    f.write(f"  Range: {df_merged['RH_Mean'].min():.2f}% - {df_merged['RH_Mean'].max():.2f}%\n\n")

print("\n✓ Saved: analysis_summary.txt")

print("\n" + "="*80)
print("ALL ANALYSES COMPLETED SUCCESSFULLY!")
print("="*80)
print("\nFiles ready for download/use:")
print("  • 2 high-resolution visualizations (600 DPI)")
print("  • 3 data files (CSV format)")
print("  • 2 detailed text reports")
print("\nAll figures have large, legible fonts suitable for presentations and publications.")
print("="*80)

# Download files if in Colab
try:
    from google.colab import files
    print("\n" + "="*80)
    print("DOWNLOAD FILES (Uncomment lines below to download):")
    print("="*80)
    print("""
# Data files
# files.download('merged_disease_weather_data.csv')
# files.download('summary_statistics_table.csv')

# Reports
# files.download('statistical_analysis_results.txt')
# files.download('analysis_summary.txt')

# Visualizations (600 DPI)
# files.download('fig1_combined_timeseries.png')
# files.download('fig2_correlation_matrix.png')
""")
except ImportError:
    print("\nFiles saved in current directory.")
    print("Access them from the Files panel in Colab.")


✓ Libraries imported successfully!
✓ High-resolution settings (600 DPI) configured
✓ Disease data loaded
✓ Date range: 2024-09-24 to 2024-10-30
✓ Total observations: 29 days

Disease Data Summary:
  PB (Pebrine): 6 positive cases
  VR (Viriosis): 17 positive cases
  BT (Bacteriosis): 6 positive cases

Fetching weather data...
  Location: (23.3441°N, 85.3096°E)
  Date range: 2024-09-24 to 2024-10-30
✓ Successfully fetched 888 hourly records

✓ THI calculated for all hourly records

✓ Daily weather statistics calculated

✓ Disease and weather data merged
✓ Final dataset: 29 observations

✓ Merged data saved to 'merged_disease_weather_data.csv'

STATISTICAL ANALYSIS

--- Analysis for PB (Pebrine) ---
Observations: 29
Positive cases (yes): 6
Negative cases (no): 23

Correlation with THI: -0.3165
P-value: 0.0943
Significant (p<0.05): No

Mean THI when PB=Yes: 71.90
Mean THI when PB=No: 73.48
Difference: -1.57

--- Analysis for VR (Viriosis) ---
Observations: 29
Positive cases (yes): 17
Nega